# Team-level tactical conclusions: Barça vs Atlético

Second stage of the pipeline. It reads the per-clip analyses written by the main notebook (`<clip>_analysis.txt` in `lost_under_pressure` and `kept_under_pressure`), combines them into one prompt, sends it to Claude Opus 5.5 and saves a general conclusion about the strengths and weaknesses of each team.

**Run order:** the two cells below, top to bottom. The second one needs `pip install -U anthropic` and `ANTHROPIC_API_KEY`.

## 1. Build the prompt

Loads every analysis file, labels it with its team (BAR / ATM) and outcome (LOST / KEPT), and builds the prompt: context, task, rules and output format. The rules ask for **general** conclusions only: no numbers, clip IDs, timestamps or player names. If the clips fit in one prompt (about 100k tokens, estimated as characters / 4) it writes `prompts/final_prompt.txt`; otherwise it writes `map_prompt_NN.txt` parts plus `reduce_prompt.txt`. Prompt files from a previous run are deleted first.

In [3]:
import re
from pathlib import Path

BASE = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\new\press\buildup_output\clips_buildup")
OUT = Path("prompts"); OUT.mkdir(exist_ok=True)
# remove prompt files of a previous run, so the folder always matches this run (single prompt OR parts + reduce)
for old_f in [OUT / "final_prompt.txt", OUT / "reduce_prompt.txt", *OUT.glob("map_prompt_*.txt")]:
    old_f.unlink(missing_ok=True)
FOLDERS = {"lost_under_pressure": "LOST", "kept_under_pressure": "KEPT"}

MAX_SINGLE_TOKENS = 100_000
CHUNK_TOKENS = 40_000
est = lambda s: len(s) // 4

# LOAD ALL FILES
records = []
for folder, label in FOLDERS.items():
    for f in sorted((BASE / folder).glob("*.txt")):
        m = re.match(r"(b\d+)_([A-Z]+)_(\d\d-\d\d)", f.stem)
        cid, team, t = m.groups() if m else (f.stem, "UNK", "")
        text = f.read_text(encoding="utf-8").strip()
        # clip ID and time are kept only so the file is traceable; the rules forbid citing them
        records.append(dict(block=f"### CLIP | team={team} | outcome={label}\n{text}\n"))

CONTEXT = """You are a football tactical analyst. Below are automated analyses of build-up clips under pressure
from a Barcelona (BAR) vs Atletico Madrid (ATM) match. The team code in each clip label is assumed to be the
team building up. LOST = the ball was lost under pressure, KEPT = the ball was kept under pressure.
Each analysis has VERDICT, WHY, COUNTERFACTUAL."""

RULES = """RULES:
- Write GENERAL conclusions only. Describe tactical tendencies and principles, not individual events.
- Do NOT give percentages, counts, ratios, success/failure rates, or win/loss statistics.
- Do NOT mention clip IDs, timestamps, player names, shirt numbers, or specific moments.
- Do NOT quote the clips or describe single situations. Speak at team level (e.g. "the press tends to...").
- Use only themes supported by the clips. Do not invent facts.
- Use qualitative wording such as "often", "tends to", "rarely" instead of numbers.
- You may contrast what generally separates LOST from KEPT situations (presser count, closing speed,
  free passing options, distance of the closest defender, block compactness), again without numbers."""

FORMAT = """OUTPUT FORMAT:
1. BAR: build-up under pressure (general strengths / weaknesses); pressing (general strengths / weaknesses)
2. ATM: build-up under pressure (general strengths / weaknesses); pressing (general strengths / weaknesses)
3. What generally separates a ball lost under pressure from a ball kept
4. General tactical takeaways (3-5 bullets)
5. Limitations (what this kind of data cannot show)"""

total = sum(est(r["block"]) for r in records)
print(f"{len(records)} clips, ~{total:,} tokens")

if total <= MAX_SINGLE_TOKENS:
    prompt = (f"{CONTEXT}\n\nTASK: Read ALL clips and write a general conclusion about the strengths "
              f"and weaknesses of each team.\n\n{RULES}\n\n{FORMAT}\n\nCLIPS:\n\n"
              + "\n".join(r["block"] for r in records))
    (OUT / "final_prompt.txt").write_text(prompt, encoding="utf-8")
    print("Wrote prompts/final_prompt.txt")
else:
    parts, cur, cur_t = [], [], 0
    for r in records:
        t = est(r["block"])
        if cur and cur_t + t > CHUNK_TOKENS:
            parts.append(cur); cur, cur_t = [], 0
        cur.append(r); cur_t += t
    if cur: parts.append(cur)

    for i, p in enumerate(parts, 1):
        prompt = (f"{CONTEXT}\n\nTASK (part {i} of {len(parts)}): Identify the general recurring tactical "
                  f"themes in the clips below, per team, as strengths or weaknesses, in build-up or pressing. "
                  f"Write short general statements only.\n\n{RULES}\n\nCLIPS:\n\n"
                  + "\n".join(r["block"] for r in p))
        (OUT / f"map_prompt_{i:02d}.txt").write_text(prompt, encoding="utf-8")

    reduce = (f"{CONTEXT}\n\nBelow are general theme summaries from {len(parts)} parts of the same match.\n"
              f"TASK: Merge them into one general conclusion about each team's strengths and weaknesses. "
              f"Themes that recur across parts should be emphasized in wording, not numbers.\n\n{RULES}\n\n{FORMAT}\n\n"
              + "\n\n".join(f"--- PART {i} SUMMARY ---\n[PASTE OUTPUT OF map_prompt_{i:02d} HERE]"
                            for i in range(1, len(parts) + 1)))
    (OUT / "reduce_prompt.txt").write_text(reduce, encoding="utf-8")
    print(f"Wrote {len(parts)} map prompts + reduce_prompt.txt")

28 clips, ~15,287 tokens
Wrote prompts/final_prompt.txt


## 2. Send to Claude Opus 5.5 and save the conclusions

Sends `final_prompt.txt` to `claude-opus-5-5` (effort `high`, thinking always on) and saves the answer to `bar_atm_tactical_conclusions.txt` in the `press` folder, overwriting the previous version. In the split case it sends each part, pastes the part summaries into the reduce prompt automatically and sends that. The cost is printed.

In [ ]:
import os, time
from pathlib import Path
import anthropic            # pip install -U anthropic  (once)

OUT = Path("prompts")                           # same folder as the cell above
RESULT_PATH = Path(r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\new\press\bar_atm_tactical_conclusions.txt")

MODEL = "claude-opus-5-5"
API_KEY = os.environ.get("ANTHROPIC_API_KEY")   # set once in the notebook:  os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."   (never share a notebook with the key inside)
EFFORT = "high"                                 # "low" | "medium" | "high" | "xhigh" | "max".  Opus 5.5 ALWAYS thinks (adaptive thinking, cannot be disabled): effort controls how much
MAX_TOKENS = 32000                              # hard limit on thinking + answer together
PRICE_IN_PER_M, PRICE_OUT_PER_M = 4.00, 20.00   # USD per 1M tokens, Claude Opus 5.5 (thinking tokens are billed as output)

assert API_KEY, 'no API key: run  os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  first'
client = anthropic.Anthropic(api_key=API_KEY)
total_cost = 0.0

def ask(prompt):
    """Send one prompt to Claude Opus 5.5 and return the answer text (streamed, because MAX_TOKENS is large)."""
    global total_cost
    with client.messages.stream(
        model=MODEL,
        max_tokens=MAX_TOKENS,
        messages=[{"role": "user", "content": prompt}],
        extra_body={"output_config": {"effort": EFFORT}},      # no `thinking` argument: thinking is always on for Opus 5.5
    ) as stream:
        resp = stream.get_final_message()
    text = "".join(blk.text for blk in resp.content if blk.type == "text").strip()   # thinking blocks are skipped
    if not text:
        raise RuntimeError(f"empty answer (stop_reason={resp.stop_reason}): raise MAX_TOKENS or lower EFFORT")
    if resp.stop_reason == "max_tokens":
        print("WARNING: the answer was cut at MAX_TOKENS")
    n_in, n_out = resp.usage.input_tokens, resp.usage.output_tokens      # output_tokens includes the thinking tokens
    cost = n_in / 1e6 * PRICE_IN_PER_M + n_out / 1e6 * PRICE_OUT_PER_M
    total_cost += cost
    print(f"   {n_in:,} in + {n_out:,} out -> ${cost:.4f}")
    return text

t_start = time.time()
final_path = OUT / "final_prompt.txt"
if final_path.exists():                                       # single prompt
    print(f"sending {final_path} to {MODEL} (effort: {EFFORT})")
    answer = ask(final_path.read_text(encoding="utf-8"))
else:                                                         # split case: one call per part, then the reduce prompt
    map_files = sorted(OUT.glob("map_prompt_*.txt"))
    assert map_files, "no prompt files found in prompts/: run the cell above first"
    summaries = []
    for i, f in enumerate(map_files, 1):
        print(f"part {i}/{len(map_files)}: {f.name}")
        summaries.append(ask(f.read_text(encoding="utf-8")))
    reduce = (OUT / "reduce_prompt.txt").read_text(encoding="utf-8")
    for i, s in enumerate(summaries, 1):
        reduce = reduce.replace(f"[PASTE OUTPUT OF map_prompt_{i:02d} HERE]", s)
    print("merging the parts (reduce prompt)")
    answer = ask(reduce)

RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)
RESULT_PATH.write_text(answer, encoding="utf-8")
print(f"\nsaved to {RESULT_PATH}\ntotal cost ${total_cost:.4f} | {(time.time() - t_start) / 60:.1f} min\n")
print("=" * 110); print(answer); print("=" * 110)
